# AIT511 Machine Learning Project I - Domestic Flight Price Prediction

**Team:** DAPS (Shailesh, Anik, Dev, Praveenraj) | **TA:** Akshat Batra

## Course Topics Covered
- Linear Regression (Simple, Multiple, Polynomial, Multivariate)
- MLE & Impact of Noise
- Regularization (Ridge, Lasso, ElasticNet)
- Gradient Descent Methods (Full, Stochastic, Mini-batch)
- Decision Trees (Regression) with & without Pruning
- EDA, Data Splitting, Cross-Validation
- Testing & Demonstrating Different Error Types
- Bias-Variance Tradeoff

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.base import BaseEstimator, TransformerMixin
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. Exploratory Data Analysis (EDA)

In [ ]:
train = pd.read_csv('flight/train.csv')
test = pd.read_csv('flight/test.csv')
sample_sub = pd.read_csv('flight/sample_submission.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Sample submission shape: {sample_sub.shape}")
print(f"\nTrain columns: {list(train.columns)}")
print(f"Test columns: {list(test.columns)}")
print(f"\nMissing values:\n{train.isnull().sum()}")

In [ ]:
# Target distribution
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0,0].hist(train['price'], bins=50, edgecolor='black', alpha=0.7)
axes[0,0].set_title('Price Distribution')
axes[0,0].set_xlabel('Price')
axes[0,0].set_ylabel('Frequency')

axes[0,1].hist(np.log1p(train['price']), bins=50, edgecolor='black', alpha=0.7)
axes[0,1].set_title('Log(Price+1) Distribution')
axes[0,1].set_xlabel('Log Price')
axes[0,1].set_ylabel('Frequency')

train['duration'].hist(bins=50, ax=axes[1,0], edgecolor='black', alpha=0.7)
axes[1,0].set_title('Duration Distribution')
axes[1,0].set_xlabel('Duration')

train['days_left'].hist(bins=50, ax=axes[1,1], edgecolor='black', alpha=0.7)
axes[1,1].set_title('Days Left Distribution')
axes[1,1].set_xlabel('Days Left')

plt.tight_layout()
plt.show()

In [ ]:
# Categorical feature analysis
cat_cols = ['airline', 'source_city', 'departure_time', 'stops', 
            'arrival_time', 'destination_city', 'flight_class']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    if i < len(axes):
        train[col].value_counts().plot(kind='bar', ax=axes[i], edgecolor='black')
        axes[i].set_title(f'{col} Distribution')
        axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Price by categorical features
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    if i < len(axes):
        train.boxplot(column='price', by=col, ax=axes[i])
        axes[i].set_title(f'Price by {col}')
        axes[i].tick_params(axis='x', rotation=45)
        axes[i].set_xlabel('')

plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix for numeric features
numeric_cols = train.select_dtypes(include=[np.number]).columns
corr_matrix = train[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 2. Data Preprocessing

In [ ]:
class FlightPreprocessor(BaseEstimator, TransformerMixin):
    """Custom preprocessor for flight data"""
    def __init__(self):
        self.cat_cols = ["airline", "source_city", "departure_time", "stops",
                         "arrival_time", "destination_city", "flight_class"]
        self.num_cols = ["duration", "days_left"]
        self.encoders = {}
        self.duration_median = None
        self.airline_mode = None
    
    def fit(self, X, y=None):
        self.duration_median = X["duration"].median()
        self.airline_mode = X["airline"].mode()[0]
        
        # Fit encoders for each categorical column
        for col in self.cat_cols:
            self.encoders[col] = {cat: i for i, cat in enumerate(X[col].unique())}
        return self
    
    def transform(self, X, train_df=None):
        X = X.copy()
        
        # Fill missing values
        X["duration"] = X["duration"].fillna(self.duration_median)
        X["airline"] = X["airline"].fillna(self.airline_mode)
        
        # Time-based features (create BEFORE encoding departure_time/arrival_time)
        time_order = {"Early_Morning": 0, "Morning": 1, "Afternoon": 2, 
                     "Evening": 3, "Night": 4, "Late_Night": 5}
        X["departure_time_num"] = X["departure_time"].map(time_order)
        X["arrival_time_num"] = X["arrival_time"].map(time_order)
        
        # Route feature
        X["route"] = X["source_city"] + "_" + X["destination_city"]
        
        # Encode categorical variables
        for col in self.cat_cols:
            X[col] = X[col].map(self.encoders[col]).fillna(-1)
        
        # Feature engineering
        X["stops_num"] = X["stops"].map({"zero": 0, "one": 1, "two": 2, "three": 3}).fillna(0)
        
        # Route encoding
        if "route" not in self.encoders:
            self.encoders["route"] = {cat: i for i, cat in enumerate(X["route"].unique())}
        X["route_encoded"] = X["route"].map(self.encoders["route"]).fillna(-1)
        
        # Drop original categorical columns and id
        drop_cols = ["id", "flight"] + self.cat_cols + ["stops", "route"]
        X = X.drop(columns=[c for c in drop_cols if c in X.columns])
        
        return X

# Apply preprocessing
preprocessor = FlightPreprocessor()
X_train = preprocessor.fit_transform(train.drop("price", axis=1))
y_train = train["price"].values
X_test = preprocessor.transform(test)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Features: {list(X_train.columns)}")

In [ ]:
# Train/Validation split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# Standardize features for linear models
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_tr.shape}, Val: {X_val.shape}")

## 3. Linear Regression from Scratch (OLS)

In [ ]:
class LinearRegressionOLS:
    """Linear Regression using Ordinary Least Squares (closed form)"""
    def __init__(self):
        self.weights = None
        self.intercept = None
    
    def fit(self, X, y):
        # Add bias term
        X_b = np.c_[np.ones(X.shape[0]), X]
        # Closed form solution: w = (X^T X)^-1 X^T y
        self.weights = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y
        self.intercept = self.weights[0]
        self.weights = self.weights[1:]
        return self
    
    def predict(self, X):
        return X @ self.weights + self.intercept

# Train OLS
ols = LinearRegressionOLS()
ols.fit(X_tr_scaled, y_tr)
y_pred_ols = ols.predict(X_val_scaled)

mse_ols = mean_squared_error(y_val, y_pred_ols)
mae_ols = mean_absolute_error(y_val, y_pred_ols)
r2_ols = r2_score(y_val, y_pred_ols)

print(f"OLS Results:")
print(f"  MSE: {mse_ols:.2f}")
print(f"  MAE: {mae_ols:.2f}")
print(f"  R²:  {r2_ols:.4f}")

## 4. Polynomial Regression

In [ ]:
# Polynomial features
poly = PolynomialFeatures(degree=2, include_bias=False)
X_tr_poly = poly.fit_transform(X_tr_scaled)
X_val_poly = poly.transform(X_val_scaled)
X_test_poly = poly.transform(X_test_scaled)

print(f"Polynomial features shape: {X_tr_poly.shape}")

# OLS on polynomial features
ols_poly = LinearRegressionOLS()
ols_poly.fit(X_tr_poly, y_tr)
y_pred_poly = ols_poly.predict(X_val_poly)

mse_poly = mean_squared_error(y_val, y_pred_poly)
mae_poly = mean_absolute_error(y_val, y_pred_poly)
r2_poly = r2_score(y_val, y_pred_poly)

print(f"Polynomial Regression (deg=2) Results:")
print(f"  MSE: {mse_poly:.2f}")
print(f"  MAE: {mae_poly:.2f}")
print(f"  R²:  {r2_poly:.4f}")

## 5. Regularized Linear Models (Ridge, Lasso, ElasticNet)

In [ ]:
# Ridge Regression with CV
ridge = Ridge()
alphas = np.logspace(-3, 3, 20)
cv_scores = []

for alpha in alphas:
    ridge.set_params(alpha=alpha)
    scores = cross_val_score(ridge, X_tr_scaled, y_tr, cv=5, scoring='neg_mean_squared_error')
    cv_scores.append(-scores.mean())

best_alpha = alphas[np.argmin(cv_scores)]
print(f"Best Ridge alpha: {best_alpha:.4f}")

ridge_best = Ridge(alpha=best_alpha)
ridge_best.fit(X_tr_scaled, y_tr)
y_pred_ridge = ridge_best.predict(X_val_scaled)

mse_ridge = mean_squared_error(y_val, y_pred_ridge)
print(f"Ridge MSE: {mse_ridge:.2f}")

# Lasso with CV
lasso = Lasso(max_iter=10000)
cv_scores_lasso = []

for alpha in alphas:
    lasso.set_params(alpha=alpha)
    scores = cross_val_score(lasso, X_tr_scaled, y_tr, cv=5, scoring='neg_mean_squared_error')
    cv_scores_lasso.append(-scores.mean())

best_alpha_lasso = alphas[np.argmin(cv_scores_lasso)]
print(f"Best Lasso alpha: {best_alpha_lasso:.4f}")

lasso_best = Lasso(alpha=best_alpha_lasso, max_iter=10000)
lasso_best.fit(X_tr_scaled, y_tr)
y_pred_lasso = lasso_best.predict(X_val_scaled)

mse_lasso = mean_squared_error(y_val, y_pred_lasso)
print(f"Lasso MSE: {mse_lasso:.2f}")

## 6. Gradient Descent Methods (Full, Stochastic, Mini-batch)

In [ ]:
class LinearRegressionGD:
    """Linear Regression with Gradient Descent variants"""
    def __init__(self, lr=0.01, epochs=1000, batch_size=None, verbose=False):
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size  # None=Full GD, 1=SGD, >1=Mini-batch
        self.verbose = verbose
        self.weights = None
        self.bias = None
        self.loss_history = []
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        # Initialize weights
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        if self.batch_size is None:
            self.batch_size = n_samples  # Full batch
        
        for epoch in range(self.epochs):
            # Shuffle data for SGD/Mini-batch
            if self.batch_size < n_samples:
                indices = np.random.permutation(n_samples)
                X_shuffled = X[indices]
                y_shuffled = y[indices]
            else:
                X_shuffled = X
                y_shuffled = y
            
            epoch_loss = 0
            n_batches = 0
            
            for i in range(0, n_samples, self.batch_size):
                X_batch = X_shuffled[i:i+self.batch_size]
                y_batch = y_shuffled[i:i+self.batch_size]
                
                # Predictions
                y_pred = X_batch @ self.weights + self.bias
                
                # Gradients
                dw = (2 / len(y_batch)) * X_batch.T @ (y_pred - y_batch)
                db = (2 / len(y_batch)) * np.sum(y_pred - y_batch)
                
                # Update
                self.weights -= self.lr * dw
                self.bias -= self.lr * db
                
                # Loss
                batch_loss = np.mean((y_pred - y_batch) ** 2)
                epoch_loss += batch_loss
                n_batches += 1
            
            avg_loss = epoch_loss / n_batches
            self.loss_history.append(avg_loss)
            
            if self.verbose and epoch % 100 == 0:
                print(f"Epoch {epoch}: Loss = {avg_loss:.4f}")
        
        return self
    
    def predict(self, X):
        return X @ self.weights + self.bias

# Compare GD variants
print("=== Full Batch GD ===")
gd_full = LinearRegressionGD(lr=0.01, epochs=500, batch_size=None, verbose=True)
gd_full.fit(X_tr_scaled, y_tr)
y_pred_full = gd_full.predict(X_val_scaled)
print(f"MSE: {mean_squared_error(y_val, y_pred_full):.2f}")

print("\n=== Stochastic GD ===")
gd_sgd = LinearRegressionGD(lr=0.01, epochs=50, batch_size=1, verbose=True)
gd_sgd.fit(X_tr_scaled, y_tr)
y_pred_sgd = gd_sgd.predict(X_val_scaled)
print(f"MSE: {mean_squared_error(y_val, y_pred_sgd):.2f}")

print("\n=== Mini-batch GD ===")
gd_mini = LinearRegressionGD(lr=0.01, epochs=200, batch_size=32, verbose=True)
gd_mini.fit(X_tr_scaled, y_tr)
y_pred_mini = gd_mini.predict(X_val_scaled)
print(f"MSE: {mean_squared_error(y_val, y_pred_mini):.2f}")

In [ ]:
# Plot loss curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(gd_full.loss_history)
axes[0].set_title('Full Batch GD Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')

axes[1].plot(gd_sgd.loss_history)
axes[1].set_title('Stochastic GD Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE')

axes[2].plot(gd_mini.loss_history)
axes[2].set_title('Mini-batch GD Loss')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('MSE')

plt.tight_layout()
plt.show()

## 7. Decision Tree Regression with Pruning

In [ ]:
# Decision Tree without pruning
dt_no_prune = DecisionTreeRegressor(random_state=42)
dt_no_prune.fit(X_tr_scaled, y_tr)
y_pred_dt = dt_no_prune.predict(X_val_scaled)
mse_dt = mean_squared_error(y_val, y_pred_dt)
print(f"Decision Tree (no pruning) MSE: {mse_dt:.2f}")
print(f"Tree depth: {dt_no_prune.get_depth()}")
print(f"Number of leaves: {dt_no_prune.get_n_leaves()}")

# Decision Tree with pre-pruning (max_depth)
for max_depth in [3, 5, 7, 10, 15, 20]:
    dt = DecisionTreeRegressor(max_depth=max_depth, random_state=42)
    dt.fit(X_tr_scaled, y_tr)
    y_pred = dt.predict(X_val_scaled)
    mse = mean_squared_error(y_val, y_pred)
    print(f"max_depth={max_depth}: MSE={mse:.2f}, depth={dt.get_depth()}, leaves={dt.get_n_leaves()}")

In [ ]:
# Cost-Complexity Pruning (post-pruning)
dt_full = DecisionTreeRegressor(random_state=42)
dt_full.fit(X_tr_scaled, y_tr)

path = dt_full.cost_complexity_pruning_path(X_tr_scaled, y_tr)
ccp_alphas = path.ccp_alphas
impurities = path.impurities

print(f"Number of ccp_alphas: {len(ccp_alphas)}")
print(f"Last few alphas: {ccp_alphas[-5:]}")

# Evaluate each alpha on validation set
mse_pruned = []
for alpha in ccp_alphas[:-1]:  # Skip the last one (empty tree)
    dt_pruned = DecisionTreeRegressor(random_state=42, ccp_alpha=alpha)
    dt_pruned.fit(X_tr_scaled, y_tr)
    y_pred = dt_pruned.predict(X_val_scaled)
    mse_pruned.append(mean_squared_error(y_val, y_pred))

best_idx = np.argmin(mse_pruned)
best_alpha = ccp_alphas[best_idx]
print(f"\nBest ccp_alpha: {best_alpha:.6f}")
print(f"Best MSE: {mse_pruned[best_idx]:.2f}")

In [ ]:
# Train final pruned tree
dt_best = DecisionTreeRegressor(random_state=42, ccp_alpha=best_alpha)
dt_best.fit(X_tr_scaled, y_tr)
y_pred_best = dt_best.predict(X_val_scaled)
mse_best = mean_squared_error(y_val, y_pred_best)

print(f"Pruned Tree: MSE={mse_best:.2f}, depth={dt_best.get_depth()}, leaves={dt_best.get_n_leaves()}")

# Compare all models
results = {
    'OLS': mse_ols,
    'Polynomial (deg=2)': mse_poly,
    'Ridge': mse_ridge,
    'Lasso': mse_lasso,
    'Full GD': mean_squared_error(y_val, y_pred_full),
    'SGD': mean_squared_error(y_val, y_pred_sgd),
    'Mini-batch GD': mean_squared_error(y_val, y_pred_mini),
    'Decision Tree (no prune)': mse_dt,
    'Decision Tree (pruned)': mse_best,
}

print("\n=== Model Comparison (Validation MSE) ===")
for name, mse in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name:30s}: {mse:.2f}")

## 8. Cross-Validation & Bias-Variance Analysis

In [ ]:
# 5-Fold Cross Validation for best models
kf = KFold(n_splits=5, shuffle=True, random_state=42)

models_cv = {
    'OLS': LinearRegression(),
    'Ridge': Ridge(alpha=best_alpha),
    'Lasso': Lasso(alpha=best_alpha_lasso, max_iter=10000),
    'Decision Tree (pruned)': DecisionTreeRegressor(random_state=42, ccp_alpha=best_alpha),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
}

cv_results = {}
for name, model in models_cv.items():
    scores = cross_val_score(model, X_tr_scaled, y_tr, cv=kf, scoring='neg_mean_squared_error')
    cv_results[name] = (-scores.mean(), scores.std())
    print(f"{name:30s}: MSE = {-scores.mean():.2f} ± {scores.std():.2f}")

In [ ]:
# Learning curves for bias-variance analysis
from sklearn.model_selection import learning_curve

train_sizes = np.linspace(0.1, 1.0, 10)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, model) in zip(axes, [('Ridge', Ridge(alpha=best_alpha)), 
                                     ('Decision Tree', DecisionTreeRegressor(random_state=42, ccp_alpha=best_alpha)),
                                     ('Random Forest', RandomForestRegressor(n_estimators=100, random_state=42))]):
    train_sizes_abs, train_scores, val_scores = learning_curve(
        model, X_tr_scaled, y_tr, cv=5, train_sizes=train_sizes,
        scoring='neg_mean_squared_error', random_state=42
    )
    
    train_mean = -train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = -val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)
    
    ax.plot(train_sizes_abs, train_mean, 'o-', label='Training MSE')
    ax.fill_between(train_sizes_abs, train_mean - train_std, train_mean + train_std, alpha=0.1)
    ax.plot(train_sizes_abs, val_mean, 'o-', label='Validation MSE')
    ax.fill_between(train_sizes_abs, val_mean - val_std, val_mean + val_std, alpha=0.1)
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('MSE')
    ax.set_title(f'{name} Learning Curve')
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

## 9. Ensemble Methods & Final Model Selection

In [ ]:
# Train final ensemble
from sklearn.ensemble import StackingRegressor

estimators = [
    ('ridge', Ridge(alpha=best_alpha)),
    ('lasso', Lasso(alpha=best_alpha_lasso, max_iter=10000)),
    ('dt', DecisionTreeRegressor(random_state=42, ccp_alpha=best_alpha)),
    ('rf', RandomForestRegressor(n_estimators=200, random_state=42)),
    ('gb', GradientBoostingRegressor(n_estimators=200, random_state=42)),
]

stacking = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=1.0),
    cv=5
)

stacking.fit(X_tr_scaled, y_tr)
y_pred_stack = stacking.predict(X_val_scaled)
mse_stack = mean_squared_error(y_val, y_pred_stack)

print(f"Stacking Ensemble MSE: {mse_stack:.2f}")

## 10. Generate Submission

In [ ]:
# Retrain on full training data
X_train_scaled = scaler.fit_transform(X_train)
stacking.fit(X_train_scaled, y_train)

# Predict on test set
test_pred = stacking.predict(X_test_scaled)

# Ensure positive prices
test_pred = np.maximum(test_pred, 0)

# Create submission
submission = pd.DataFrame({
    'id': test['id'],
    'price': test_pred
})

print(f"Submission shape: {submission.shape}")
print(submission.head())

submission.to_csv('flight/submission.csv', index=False)
print("Saved to flight/submission.csv")

In [ ]:
# Verify submission format
sample = pd.read_csv('flight/sample_submission.csv')
print(f"Sample submission shape: {sample.shape}")
print(f"Our submission shape: {submission.shape}")
print(f"Columns match: {list(sample.columns) == list(submission.columns)}")
print(f"ID match: {(sample['id'] == submission['id']).all()}")